In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. 분석 설정 (파라미터 세팅)
# ---------------------------------------------------------
DATASET = 'COPERNICUS/S5P/NRTI/L3_NO2'
BAND = 'tropospheric_NO2_column_number_density'
YEAR = 2023
MONTHS = [3, 4, 5]
MIN_QA_VALUE = 0.75
USE_COMMON_PIXELS = True

# 출력 폴더 설정
OUT = Path('/content/seoul_no2_results') / f'qa{MIN_QA_VALUE}_common{int(USE_COMMON_PIXELS)}'
OUT.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2. JUSO 경계 데이터 불러오기 (서울시 자치구 단위)
# ---------------------------------------------------------
# 주의: 'seoul_sigungu.shp' 등 서울시 구 경계 Shapefile 업로드 필요
file_path = '/content/seoul_sigungu.shp' 
seoul_gdf = gpd.read_file(file_path, encoding='euc-kr').to_crs(epsg=4326)

# GEE FeatureCollection으로 변환
seoul_ee = geemap.geopandas_to_ee(seoul_gdf)

# ---------------------------------------------------------
# 3. 월별 NO2 데이터 및 유효 픽셀(Mask) 추출 함수
# ---------------------------------------------------------
def get_monthly_mean_and_mask(year, month, roi):
    start_date = ee.Date.fromYMD(year, month, 1)
    end_date = start_date.advance(1, 'month')
    
    # 1) 해당 월의 데이터 컬렉션 불러오기
    col = ee.ImageCollection(DATASET) \
            .filterBounds(roi) \
            .filterDate(start_date, end_date)
            
    # 2) QA 필터링 함수
    def apply_qa(img):
        qa = img.select('qa_value')
        mask = qa.gt(MIN_QA_VALUE)
        return img.updateMask(mask).select(BAND)
    
    # 3) QA 통과한 데이터의 한 달 평균값 계산
    monthly_mean = col.map(apply_qa).mean()
    
    # 4) 유효 픽셀 마스크 추출 (데이터가 있으면 1, 없으면 0)
    monthly_mask = monthly_mean.mask()
    
    return monthly_mean, monthly_mask

# ---------------------------------------------------------
# 4. 공통 픽셀(교집합) 찾기 로직 (핵심)
# ---------------------------------------------------------
images = {}
masks = []

# 각 월별로 이미지와 마스크 수집
for m in MONTHS:
    img, mask = get_monthly_mean_and_mask(YEAR, m, seoul_ee)
    images[m] = img
    masks.append(mask)

# 3, 4, 5월의 마스크를 교집합(And 연산) 처리하여 '공통 픽셀' 추출
common_mask = masks[0]
for m_mask in masks[1:]:
    common_mask = common_mask.And(m_mask) # 3개월 모두 1(유효)인 픽셀만 남김

# ---------------------------------------------------------
# 5. 각 월별 데이터에 공통 픽셀 마스크 씌우고 자치구별 평균 추출
# ---------------------------------------------------------
results = []

for m in MONTHS:
    # 해당 월 이미지에 3개월 공통 마스크 적용 (중요!)
    final_img = images[m].updateMask(common_mask).clip(seoul_ee)
    
    # 자치구(구 단위)별 평균값 계산 (Zonal Statistics)
    # seoul_ee 경계 내부에 있는 픽셀들의 평균을 계산
    stats = final_img.reduceRegions(
        collection=seoul_ee,
        reducer=ee.Reducer.mean(),
        scale=1113.2 # Sentinel-5P 픽셀 해상도 (약 1.1km)
    ).getInfo() # 결과를 파이썬 딕셔너리로 가져오기
    
    # 결과 정리
    for feature in stats['features']:
        # 'SIG_KOR_NM' 등 자치구 이름이 들어있는 컬럼명 확인 필요
        gu_name = feature['properties'].get('SIG_KOR_NM', 'Unknown') 
        mean_no2 = feature['properties'].get('mean', None)
        
        results.append({
            'Year': YEAR,
            'Month': m,
            'District': gu_name,
            'NO2_Mean': mean_no2
        })

# ---------------------------------------------------------
# 6. 결과를 엑셀(CSV)로 저장하고 출력
# ---------------------------------------------------------
df = pd.DataFrame(results)
output_file = OUT / f'seoul_no2_{YEAR}_spring_common.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"분석 완료! 결과가 저장되었습니다: {output_file}")
display(df.head(10)) # 분석 결과 10줄 미리보기


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. 파일 경로 설정 (이전 단계에서 저장한 CSV 파일 경로)
input_file = '/content/seoul_no2_results/qa0.75_common1/seoul_no2_2019_2020_spring_common.csv'

# 2. 데이터 불러오기
df = pd.read_csv(input_file)

# 3. 2019년, 2020년 데이터만 추출
df = df[df['Year'].isin([2019, 2020])]

# 4. 데이터 형태 변환 (Pivot)
# 자치구(District)와 월(Month)을 고정하고, 연도(Year)를 열(Column)로 펼칩니다.
pivot_df = df.pivot_table(
    index=['District', 'Month'], 
    columns='Year', 
    values='NO2_Mean'
).reset_index()

# 피벗된 컬럼명 직관적으로 변경
pivot_df.rename(columns={2019: 'NO2_2019', 2020: 'NO2_2020'}, inplace=True)

# 5. 증감률(%) 계산
# 공식: ((비교연도 - 기준연도) / 기준연도) * 100
# 조건: BASELINE_MIN(0.0) 이하인 경우 변화율을 계산하지 않고 결측치(NaN)로 처리
BASELINE_MIN = 0.0

pivot_df['Change_Rate(%)'] = np.where(
    pivot_df['NO2_2019'] > BASELINE_MIN, 
    ((pivot_df['NO2_2020'] - pivot_df['NO2_2019']) / pivot_df['NO2_2019']) * 100,
    np.nan
)

# 소수점 둘째 자리까지 반올림하여 깔끔하게 정리 (선택 사항)
pivot_df['Change_Rate(%)'] = pivot_df['Change_Rate(%)'].round(2)

# 6. 결과 저장
output_dir = Path('/content/seoul_no2_results')
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / 'seoul_no2_2019_vs_2020_change_rate.csv'
pivot_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"증감률 계산 완료! 파일이 저장되었습니다: {output_file}")
display(pivot_df) # 결과 표 출력

In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
from pathlib import Path

# 1. 설정값 (월별로 결측치 비율이 다름을 가정하므로 USE_COMMON_PIXELS는 False 방식 적용)
DATASET = 'COPERNICUS/S5P/NRTI/L3_NO2'
BAND = 'tropospheric_NO2_column_number_density'
YEAR = 2023
MONTHS = [3, 4, 5]
MIN_QA_VALUE = 0.75

# 서울시 JUSO 경계 불러오기 (앞선 코드와 동일)
file_path = '/content/seoul_sigungu.shp' 
seoul_gdf = gpd.read_file(file_path, encoding='euc-kr').to_crs(epsg=4326)
seoul_ee = geemap.geopandas_to_ee(seoul_gdf)

results = []

# 2. 월별 평균(mean) 및 유효 픽셀 수(count) 동시 추출
for m in MONTHS:
    start_date = ee.Date.fromYMD(YEAR, m, 1)
    end_date = start_date.advance(1, 'month')
    
    # 데이터 로드 및 QA 필터링
    col = ee.ImageCollection(DATASET).filterBounds(seoul_ee).filterDate(start_date, end_date)
    
    def apply_qa(img):
        mask = img.select('qa_value').gt(MIN_QA_VALUE)
        return img.updateMask(mask).select(BAND)
    
    monthly_mean = col.map(apply_qa).mean().clip(seoul_ee)
    
    # [핵심] Reducer를 결합하여 평균(mean)과 유효 픽셀 개수(count)를 한 번에 계산
    combined_reducer = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.count(),
        sharedInputs=True
    )
    
    stats = monthly_mean.reduceRegions(
        collection=seoul_ee,
        reducer=combined_reducer,
        scale=1113.2
    ).getInfo()
    
    # 리스트에 저장
    for feature in stats['features']:
        gu_name = feature['properties'].get('SIG_KOR_NM', 'Unknown')
        mean_no2 = feature['properties'].get('mean', np.nan)
        pixel_count = feature['properties'].get('count', 0) # 결측치 제외 살아남은 픽셀 수
        
        results.append({
            'District': gu_name,
            'Month': m,
            'NO2_Mean': mean_no2,
            'Valid_Pixels': pixel_count
        })

# 3. Pandas를 활용한 가중평균 계산
df = pd.DataFrame(results)

# 결측치(관측 데이터가 0개인 경우) 제거
df = df.dropna(subset=['NO2_Mean'])

# [가중치 연산 1] 각 월별 '평균 농도 × 유효 픽셀 수' 계산 (총량 구하기)
df['Weighted_Sum'] = df['NO2_Mean'] * df['Valid_Pixels']

# [가중치 연산 2] 자치구별로 3개월 치 총량과 총 유효 픽셀 수를 합산
weighted_df = df.groupby('District').agg(
    Total_Weighted_Sum=('Weighted_Sum', 'sum'),
    Total_Valid_Pixels=('Valid_Pixels', 'sum')
).reset_index()

# [가중치 연산 3] 총량을 총 유효 픽셀 수로 나누어 최종 3개월 가중평균 산출
weighted_df['3Month_Weighted_Avg'] = weighted_df['Total_Weighted_Sum'] / weighted_df['Total_Valid_Pixels']

# 결과 확인
print("🌸 자치구별 3개월(봄철) 가중평균 결과 🌸")
display(weighted_df[['District', 'Total_Valid_Pixels', '3Month_Weighted_Avg']])

# 필요시 CSV 저장
output_file = Path('/content/seoul_no2_results/seoul_no2_weighted_avg.csv')
output_file.parent.mkdir(parents=True, exist_ok=True)
weighted_df.to_csv(output_file, index=False, encoding='utf-8-sig')